<a href="https://colab.research.google.com/github/nikitask14/pytorch-engineering-to-federated-learning/blob/main/Sitting21_manual_weight_averaging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Sitting 21 — Manual Weight Averaging / FedAvg Bridge**

##### **Objective**

Implement the mathematical heart of Federated Averaging (FedAvg) manually in pure PyTorch.

````text The complete flow is:

global model
      ↓
independent clients with the same starting state
      ↓
each client trains on its own local data
      ↓
extract client states
      ↓
aggregate corresponding parameter tensors
      ↓
create a new global state
      ↓
load the aggregated state into the global model

````

**Imports**

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

**Architecture: 2 → 4 → 1, with ReLU between the two linear layers**

We use a small neural network so that the aggregation mechanics remain visible.

**Define the model:**
The model contains two Linear layers with ReLU between them.

The model state therefore contains four learnable parameter tensors:

- layer1.weight → shape (4,2)
- layer1.bias   → shape (4,)
- layer2.weight → shape (1,4)
- layer2.bias   → shape (1,)

These corresponding tensors will later be aggregated across clients.

In [2]:
class MyModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(2,4)
    self.layer2 = nn.Linear(4,1)

  def forward(self, x):
    x = torch.relu(self.layer1(x))
    x = self.layer2(x)

    return x



 **Create Global Model, and Client1 and Client 2 Initialisation**

 Federated-learning clients require two things:

1. Different model objects.
2. The same starting parameter values.


In [3]:
#torch.manual_seed(42) makes the initial random conditions reproducible.
torch.manual_seed(42)

#Create global model and initialise client1 and client2
global_model = MyModel()
client1 = MyModel()
client2 = MyModel()

# Checking if both clients are independent objects
print(global_model is client1)
print(client1 is client2)
print(client2 is global_model)

False
False
False


In [4]:
# Both clients must begin with exactly the same
# parameter values as global_model

global_model_state = global_model.state_dict()

# transfer the global parameter values into both of them.
client1.load_state_dict(global_model_state)
client2.load_state_dict(global_model_state)




<All keys matched successfully>

In [5]:
# check if both clients begin with the same parameter value ie the same statrting state
torch.equal(client1.state_dict()["layer1.weight"], client2.state_dict()["layer1.weight"])

True

**Separate local data + independent training**

In [6]:

# Client 1: 4 examples

X1 = torch.tensor([
    [1,1],
    [1,2],
    [2,1],
    [2,2]
], dtype = torch.float32)

y1 = torch.tensor([
   [3],
   [5],
   [4],
   [6]
], dtype = torch.float32)

In [7]:
# Client 2: 6 examples

X2 = torch.tensor([
   [3,1],
   [3,2],
   [4,1],
   [4,2],
   [5,1],
   [5,2]
], dtype = torch.float32)

y2 = torch.tensor([
   [5],
   [7],
   [6],
   [8],
   [7],
   [9]

], dtype = torch.float32)

**Dataset and Dataset Loader**

In [8]:
# Create TensorDataset and one DataLoader for each client.
train_dataset1 = TensorDataset(X1,y1)
train_loader1 = DataLoader(train_dataset1, batch_size = 2, shuffle = True)

train_dataset2 = TensorDataset(X2,y2)
train_loader2 = DataLoader(train_dataset2, batch_size = 2, shuffle = True )



**Loss and Sepearate Optmisation**

Both clients can use the same optimizer type, learning rate, loss function, and number of local epochs, but each optimizer must point to its own client's parameters.

In [9]:
optimiser1 = torch.optim.SGD(client1.parameters(), lr = 0.01)
optimiser2 = torch.optim.SGD(client2.parameters(), lr = 0.01)

loss_fn = nn.MSELoss()

In [10]:
# Training loop for client1
# epoch = 20
for epoch in range(20):
  client1.train()
  for x_batch, y_batch in train_loader1:
    optimiser1.zero_grad()
    prediction1 = client1(x_batch)
    train_loss1 = loss_fn(prediction1,y_batch)
    train_loss1.backward()
    optimiser1.step()


In [11]:
# Training loop for client2
for epoch in range(20):
  client2.train()
  for x_batch, y_batch in train_loader2:
    optimiser2.zero_grad()
    predictions2 = client2(x_batch)
    train_loss2 = loss_fn(predictions2, y_batch)
    train_loss2.backward()
    optimiser2.step()

In [38]:
# obtain a model's state dictionary
# extract client's state
client1_state = client1.state_dict()
client2_state = client2.state_dict()

Before averaging, we should verify two things:

1. At least some corresponding parameter tensors are now different, because the clients trained independently on different local data.
2. They contain the same parameter names.

In [13]:
# This tells us that since client1 and client2 have trained idependently
# on different local data, their corresponding parameter tensors are now different
torch.equal(client1_state["layer1.weight"], client2_state["layer1.weight"])

False

In [14]:
# Inspect client1 parameters
for name,tensor in client1.named_parameters():
  print(name, tensor)


layer1.weight Parameter containing:
tensor([[ 0.7577,  0.9221],
        [-0.1180,  0.7119],
        [-0.0455,  0.3009],
        [-0.3557,  0.4015]], requires_grad=True)
layer1.bias Parameter containing:
tensor([ 0.7281, -0.4970,  0.6704,  0.1242], requires_grad=True)
layer2.weight Parameter containing:
tensor([[1.0586, 0.2343, 0.4218, 0.0200]], requires_grad=True)
layer2.bias Parameter containing:
tensor([0.5832], requires_grad=True)


In [15]:
# Inspect client2 parameters
for name,tensor in client2.named_parameters():
  print(name, tensor)


layer1.weight Parameter containing:
tensor([[ 0.7895,  1.0065],
        [-0.0460,  0.7285],
        [-0.0082,  0.2982],
        [-0.3443,  0.4153]], requires_grad=True)
layer1.bias Parameter containing:
tensor([ 0.6757, -0.4911,  0.6495,  0.1323], requires_grad=True)
layer2.weight Parameter containing:
tensor([[ 1.1754,  0.2418,  0.3521, -0.0706]], requires_grad=True)
layer2.bias Parameter containing:
tensor([0.5429], requires_grad=True)


In [16]:
# Alternative appoach to check that both clients contain
# the same parameter names using the state_dict()
print(client1_state.keys())
print(client2_state.keys())

odict_keys(['layer1.weight', 'layer1.bias', 'layer2.weight', 'layer2.bias'])
odict_keys(['layer1.weight', 'layer1.bias', 'layer2.weight', 'layer2.bias'])


#####**Manually average corresponding tensors**

In [17]:
layer1_weight = (client1_state["layer1.weight"] + client2_state["layer1.weight"])/2
layer1_weight

tensor([[ 0.7736,  0.9643],
        [-0.0820,  0.7202],
        [-0.0268,  0.2995],
        [-0.3500,  0.4084]])

**Equal-average the complete state**

In [18]:
# create an empty dictionary that will hold the new global state:
averaged_state = {}
# This is equal average weightage where both clients are contributing equally.
for name in client1_state:
    averaged_state[name] = (client1_state[name] + client2_state[name]) / 2

name = "layer1.weight"

→ average both clients' layer1.weight

→ store under averaged_state["layer1.weight"]
___

name = "layer1.bias"

→ average both clients' layer1.bias

→ store under averaged_state["layer1.bias"]

So averaged_state becomes a complete new state dictionary containing the averaged version of every corresponding tensor.


In [19]:
averaged_state

{'layer1.weight': tensor([[ 0.7736,  0.9643],
         [-0.0820,  0.7202],
         [-0.0268,  0.2995],
         [-0.3500,  0.4084]]),
 'layer1.bias': tensor([ 0.7019, -0.4941,  0.6600,  0.1283]),
 'layer2.weight': tensor([[ 1.1170,  0.2381,  0.3870, -0.0253]]),
 'layer2.bias': tensor([0.5630])}

In [23]:
# Load averaged state into global model
global_model.load_state_dict(averaged_state)
global_model.state_dict()["layer1.weight"]

tensor([[ 0.7736,  0.9643],
        [-0.0820,  0.7202],
        [-0.0268,  0.2995],
        [-0.3500,  0.4084]])

In [24]:
#verify
torch.equal(global_model.state_dict()["layer1.weight"], averaged_state["layer1.weight"])

True

**Sample-size-weighted FedAvg**

In [35]:
#Creating empty dictionary to store the state dictionary for federated averagee
# i.e. weighted average of clinet 1 and client 2
n1 = 4 #No. of examples available with CLient1
n2 = 6 #No. of examples available with CLient2
fed_avg = {}
for name in client1_state:
  fed_avg[name] = (
      (n1/(n1+n2) * client1_state[name])

                   +
      (n2/(n1+n2) * client2_state[name])
   )

fed_avg["layer1.weight"]


tensor([[ 0.7768,  0.9727],
        [-0.0748,  0.7218],
        [-0.0231,  0.2993],
        [-0.3488,  0.4098]])

Writing the stand alone aggregation function

In [37]:
# first define what the function does, then call it,
# then use its returned state to update the global model.
def fed_avg(client1_state, client2_state, n1, n2):
  fed_avg_state = {}
  for name in client1_state:
    fed_avg_state[name] = (
      (n1/(n1+n2) * client1_state[name])

                   +
      (n2/(n1+n2) * client2_state[name])
   )

  return fed_avg_state
new_global_state = fed_avg(
    client1_state,
    client2_state,
    n1,
    n2
)
global_model.load_state_dict(new_global_state)

<All keys matched successfully>

## Sitting 21 — Final Mental Model

## What happened in one federated-learning round?

### 1. Start with one global model

The server/global side owns a model with some parameter state:

θ_global

### 2. Create independent clients

Each client must have:

**Different objects + same starting state.**

The same global parameter values are transferred to each independent client.

### 3. Train locally

Each client trains only on its own local data.

Because the local datasets differ:

same starting model
      ↓
different local data
      ↓
different gradients
      ↓
different parameter updates
      ↓
different client states

### 4. Extract states

Each trained client produces a state dictionary containing corresponding
parameter tensors such as:

layer1.weight
layer1.bias
layer2.weight
layer2.bias

### 5. Aggregate corresponding tensors

We do NOT average model objects directly.

We match parameters by name and aggregate the corresponding tensors.

Equal averaging:

θ_global = (θ1 + θ2) / 2

Sample-size-weighted FedAvg:

θ_global =
(n1/(n1+n2)) θ1
+
(n2/(n1+n2)) θ2

### 6. Construct a new global state

The aggregated tensors are stored under the same parameter names in a new
state dictionary.

### 7. Load the new global state

`load_state_dict()` transfers the aggregated state into the global model.

The global model now represents information combined from the locally trained
clients.

## Core FedAvg flow

global model
      ↓
same starting state sent to independent clients
   ↙                              ↘
Client 1                         Client 2
local data                       local data
local training                   local training
   ↘                              ↙
client1_state                 client2_state
          ↘                  ↙
          aggregate corresponding
             parameter tensors
                    ↓
              new global state
                    ↓
           load into global model

## Sitting 20 → Sitting 21 connection

Sitting 20:

**Different objects + same starting state.**

Sitting 21:

**Different local training → different client states → aggregate states → new global state.**

## Most important implementation insight

`state_dict()` is the bridge.

It lets us move from:

PyTorch model objects

to:

named parameter tensors that can be inspected, copied, saved, transferred,
and mathematically aggregated.

That is why `state_dict()` appears repeatedly in model saving, checkpoints,
client initialization, and FedAvg.